# Building a Local Agent with Strands Agents and Ollama Model

This notebook demonstrates how to create a personal agent using Strands Agent and Ollama. The agent will be capable of performing various local tasks such as file operations, web searches, and system commands.

## What is Ollama?

[Ollama](https://ollama.com/) is an open-source framework that allows you to run large language models (LLMs) locally on your machine. It provides a simple API for interacting with these models, making it ideal for privacy-focused applications where you don't want to send data to external services.

Key benefits of Ollama:
- **Privacy**: All processing happens locally on your machine
- **No API costs**: Free to use as much as you want
- **Offline capability**: Works without internet connection
- **Customization**: Can be fine-tuned for specific use 


## Agent Details

<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |Ollama Model - to create a file operations agent   |
|Agent Structure     |Single agent architecture                          |


</div>


### Agent Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

In [1]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 KB 3.7 MB/s eta 0:00:00a 0:00:01


## Setup and Installation

Before running this notebook, make sure you have:

1. Install Ollama: Follow instructions at [https://ollama.com/download](https://ollama.com/download)
2. Start the Ollama server: `ollama serve`
3. Downloaded a model with Ollama: `ollama pull llama3.2:1b`

Refer to the [documentation](https://cuddly-sniffle-lrmk2y7.pages.github.io/0.1.x/user-guide/concepts/model-providers/ollama/) for detailed instructions.

In this notebook, we will download Ollama for the linux distribution for compatibility with SageMaker Studio. This is done for code execution during AWS lead workshops on Workshop Studio. If you are running this code locally, you should adjust the steps to download Ollama to your current enviroment.

In [ ]:
# this will work on linux computers
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
subprocess.Popen(['ollama', 'serve'])

In [ ]:
!ollama pull llama3.2:3b

In [2]:
# Import necessary libraries
import os

import nest_asyncio
import requests

# Import strands components
from strands import Agent, tool
from strands.models.ollama import OllamaModel

# Allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

In [3]:
# Check if Ollama is running:
try:
    response = requests.get("http://192.168.68.84:11434/api/tags")
    print("✅ Ollama is running. Available models:")
    for model in response.json().get("models", []):
        print(f"- {model['name']}")
except requests.exceptions.ConnectionError:
    print("❌ Ollama is not running. Please start Ollama before proceeding.")

✅ Ollama is running. Available models:
- nemotron-3-super:120b


## Defining Custom Tools

Tools are functions that the agent can use to interact with the environment. Below, we define several useful tools for our personal agent.

In [4]:
# File Operation Tools
@tool
def file_read(file_path: str) -> str:
    """Read a file and return its content. Supports both text and PDF files.

    Args:
        file_path (str): Path to the file to read

    Returns:
        str: Content of the file

    Raises:
        FileNotFoundError: If the file doesn't exist
    """
    try:
        # Check if it's a PDF file
        if file_path.lower().endswith('.pdf'):
            import PyPDF2
            with open(file_path, "rb") as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
                return text if text.strip() else "Error: Could not extract text from PDF"
        else:
            # Regular text file
            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()
    except FileNotFoundError:
        return f"Error: File '{file_path}' not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def file_write(file_path: str, content: str) -> str:
    """Write content to a file.

    Args:
        file_path (str): The path to the file
        content (str): The content to write to the file

    Returns:
        str: A message indicating success or failure
    """
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)

        with open(file_path, "w") as file:
            file.write(content)
        return f"File '{file_path}' written successfully."
    except Exception as e:
        return f"Error writing to file: {str(e)}"


@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories in the specified path.

    Args:
        directory_path (str): Path to the directory to list

    Returns:
        str: A formatted string listing all files and directories
    """
    try:
        items = os.listdir(directory_path)
        files = []
        directories = []

        for item in items:
            full_path = os.path.join(directory_path, item)
            if os.path.isdir(full_path):
                directories.append(f"Folder: {item}/")
            else:
                files.append(f"File: {item}")

        result = f"Contents of {os.path.abspath(directory_path)}:\n"
        result += (
            "\nDirectories:\n" + "\n".join(sorted(directories))
            if directories
            else "\nNo directories found."
        )
        result += (
            "\n\nFiles:\n" + "\n".join(sorted(files)) if files else "\nNo files found."
        )

        return result
    except Exception as e:
        return f"Error listing directory: {str(e)}"

## Creating the Ollama-powered Agent

Now we'll create our agent using the Ollama model and the tools we defined above.

Note: You can add more tools like `execute_commands`, `search_files` etc.

In [5]:
# Define a comprehensive system prompt for our agent
system_prompt = """
You are a helpful personal assistant capable of performing local file actions and simple tasks for the user.

Your key capabilities:
1. Read, understand, and summarize files.
2. Create and write to files.
3. List directory contents and provide information on the files.
4. Summarize text content

When using tools:
- Always verify file paths before operations
- Be careful with system commands
- Provide clear explanations of what you're doing
- If a task cannot be completed, explain why and suggest alternatives

Always be helpful, concise, and focus on addressing the user's needs efficiently.
"""

model_id = (
    "nemotron-3-super:120b"  # You can change this to any model you have pulled with Ollama.
)

#### Configure the Ollama model
Make sure your Ollama service is running at http://localhost:11434 and your `model_id` is in the list of Ollama models printed above.

In [6]:
ollama_model = OllamaModel(
    model_id=model_id,
    host="http://192.168.68.84:11434",
    max_tokens=4096,  # Adjust based on your model's capabilities
    temperature=0.7,  # Lower for more deterministic responses, higher for more creative
    top_p=0.9,  # Nucleus sampling parameter
)

# Create the agent
local_agent = Agent(
    system_prompt=system_prompt,
    model=ollama_model,
    tools=[file_read, file_write, list_directory],
)

## Testing the Agent

Let's test our agent with some example tasks.

In [7]:
local_agent(
    "Read the file in the path `sample_file/Amazon-com-Inc-2023-Shareholder-Letter.pdf` and summarize it in 5 bullet points."
)


Tool #1: file_read
- **Financial Performance:** Amazon’s 2023 revenue grew 12% year‑over‑year to $575 B, with operating income surging 201% to $36.9 B (margin 6.4%) and free cash flow improving from ‑$12.8 B in 2022 to $35.5 B.  
- **Customer Experience Gains:** The company broadened selection (hundreds of millions of products), sharpened pricing (customers saved nearly $24 B via deals/coupons), and achieved record delivery speeds—over 7 B items arrived same‑ or next‑day for Prime members—while reducing its global cost to serve for the first time since 2018.  
- **AWS Progress:** Cost optimization waned as new deals accelerated; AWS launched Graviton4 and Trainium2 chips, expanded to 105 Availability Zones across 33 Regions, and deepened its generative‑AI stack (SageMaker, Bedrock, Amazon Q, and custom AI chips) to empower builders.  
- **Other Business Highlights:** Advertising revenue rose 24% to $47 B; Prime Video gained traction with exclusive content, advertising, and live sports

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '- **Financial Performance:** Amazon’s 2023 revenue grew 12% year‑over‑year to $575\u202fB, with operating income surging 201% to $36.9\u202fB (margin 6.4%) and free cash flow improving from ‑$12.8\u202fB in 2022 to $35.5\u202fB.  \n- **Customer Experience Gains:** The company broadened selection (hundreds of millions of products), sharpened pricing (customers saved nearly $24\u202fB via deals/coupons), and achieved record delivery speeds—over 7\u202fB items arrived same‑ or next‑day for Prime members—while reducing its global cost to serve for the first time since 2018.  \n- **AWS Progress:** Cost optimization waned as new deals accelerated; AWS launched Graviton4 and Trainium2 chips, expanded to 105 Availability Zones across 33 Regions, and deepened its generative‑AI stack (SageMaker, Bedrock, Amazon Q, and custom AI chips) to empower builders.  \n- **Other Business Highlights:** Advertising revenu

In [8]:
# Example 2: List files in the current directory
response = local_agent("Show me the files in the current directory")


Tool #2: list_directory
Here are the files and directories in the current directory:

**Directories:**
- `images/`
- `sample_file/`

**Files:**
- `ollama_file_ops_agent.ipynb`
- `requirements.txt`

In [9]:
# Example 3: Create a sample file
response = local_agent(
    "Create a file called 'sample.txt' with the content 'This is a test file created by my Ollama agent.'"
)


Tool #3: file_write
I've successfully created the file `sample.txt` with the content: "This is a test file created by my Ollama agent."

In [10]:
# Example 4: create a readme file after reading and understanding multiple files
response = local_agent("Create a readme.md for the current directory")


Tool #4: file_write

Tool #5: list_directory
I've created a `readme.md` file in the current directory with an overview of the directory contents and usage instructions. The file has been written successfully.

Here's the updated directory listing confirming the file exists:
- Directories: `images/`, `sample_file/`
- Files: `ollama_file_ops_agent.ipynb`, `requirements.txt`, `sample.txt`, `readme.md`

## Conclusion

In this notebook, we've created a local personal agent using Stands and Ollama. The agent can perform file operations (read, write, append) and Summarize/Analyze text

This demonstrates the power of running AI models locally with Ollama, combined with the flexibility of strands' tool system. You can extend this agent by adding more tools or using different Ollama models based on your needs.

### Next Steps (Ideas)

- Try different Ollama models to see how they affect the agent's capabilities
- Add more complex tools like web search, email sending, or calendar integration
- Implement memory for the agent to remember past interactions
- Create a simple UI for interacting with your agent